# Stanford Dogs Breed Classification with ResNet-18 and ViT-B/16

This notebook is designed to support the **image track** of Assignment 1 using the **Stanford Dogs** dataset.
It covers dataset exploration (EDA), preprocessing, data loading, augmentation, transfer learning with **ResNet-18** and **ViT-B/16**, evaluation, comparison, and extension outputs for the final report.


## 0. Environment and Imports

Models:
- `ResNet-18` as the CNN baseline
- `ViT-B/16` as the Vision Transformer model

Dataset:
- official **Stanford Dogs** image classification split using `train_list.mat` and `test_list.mat`
- an internal stratified validation split created from the official training split only


In [ ]:
from pathlib import Path
import copy
import json
import math
import random
import sys
import tarfile
import time
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm as mpl_cm
from PIL import Image
from scipy.io import loadmat

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedShuffleSplit

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

sns.set_theme(style="whitegrid", context="talk", palette="colorblind")
plt.rcParams["figure.dpi"] = 140
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelweight"] = "medium"
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["legend.frameon"] = True

QUALITY_PALETTE = sns.color_palette("Set2", 6)
MODEL_PALETTE = sns.color_palette("colorblind", 2)
RGB_PALETTE = sns.color_palette("deep", 3)
RGB_COLORS = RGB_PALETTE
BRIGHTNESS_COLOR = QUALITY_PALETTE[0]
CONTRAST_COLOR = QUALITY_PALETTE[1]
SATURATION_COLOR = QUALITY_PALETTE[2]
WIDTH_COLOR = sns.color_palette("deep")[0]
HEIGHT_COLOR = sns.color_palette("deep")[1]
ASPECT_COLOR = sns.color_palette("deep")[2]
RESNET_COLOR = MODEL_PALETTE[0]
VIT_COLOR = MODEL_PALETTE[1]
HEATMAP_CMAP = "mako"
CONFUSION_CMAP = "YlGnBu"

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
def resolve_image_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        cwd / "assignments" / "assignment-1" / "image",
        cwd.parent / "assignments" / "assignment-1" / "image",
    ]
    for candidate in candidates:
        if (candidate / "models").exists() and (candidate / "artifacts").exists() and (candidate / "notebooks").exists():
            return candidate.resolve()
    return cwd.parent.resolve() if cwd.name == "notebooks" else cwd.resolve()


IMAGE_ROOT = resolve_image_root()
DATA_ROOT = IMAGE_ROOT / "data"
STANFORD_DOGS_ROOT = DATA_ROOT / "stanford_dogs"
MODEL_ROOT = IMAGE_ROOT / "models"
ARTIFACT_ROOT = IMAGE_ROOT / "artifacts" / "stanford_dogs"
EDA_ARTIFACT_ROOT = ARTIFACT_ROOT / "eda"
CNN_ARTIFACT_ROOT = ARTIFACT_ROOT / "cnn"
VIT_ARTIFACT_ROOT = ARTIFACT_ROOT / "vit"

for path in [DATA_ROOT, STANFORD_DOGS_ROOT, MODEL_ROOT, ARTIFACT_ROOT, EDA_ARTIFACT_ROOT, CNN_ARTIFACT_ROOT, VIT_ARTIFACT_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

SEED = 42
IMAGE_SIZE = 224
VAL_FROM_TRAIN_RATIO = 0.15
BATCH_SIZE = 32
NUM_WORKERS = 0 if sys.platform.startswith("win") else 4
PIN_MEMORY = torch.cuda.is_available()
PERSISTENT_WORKERS = NUM_WORKERS > 0
LOAD_EXISTING_CHECKPOINTS = True

RESNET_EPOCHS = 12
RESNET_LR = 3e-4
RESNET_WEIGHT_DECAY = 1e-4
RESNET_WARMUP_EPOCHS = 2

VIT_HEAD_EPOCHS = 3
VIT_HEAD_LR = 1e-3
VIT_FINETUNE_EPOCHS = 8
VIT_FINETUNE_LR = 3e-5
VIT_WEIGHT_DECAY = 1e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
print("IMAGE_ROOT:", IMAGE_ROOT)
print("DEVICE:", DEVICE)

RUN_MISCLASSIFIED_GALLERY = True
RUN_INTERPRETABILITY = True
RUN_ABLATIONS = False

INTERPRETABILITY_SAMPLES = 6
MISCLASSIFIED_GALLERY_SAMPLES = 12

RESNET_ABLATION_EPOCHS = 4
RESNET_FREEZE_LR = 1e-3
RESNET_FULL_LR = 3e-4

VIT_ABLATION_EPOCHS = 4
VIT_FREEZE_LR = 1e-3
VIT_FULL_LR = 3e-5


## 1. Problem and Dataset Exploration (EDA)

We will:
- download the official dataset,
- build metadata for all labeled images,
- inspect class / species / size distributions,
- create a new stratified split,
- save report-ready EDA figures.


In [ ]:
IMAGES_URL = "http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar"
LISTS_URL = "http://vision.stanford.edu/aditya86/ImageNetDogs/lists.tar"

IMAGES_ARCHIVE = STANFORD_DOGS_ROOT / "images.tar"
LISTS_ARCHIVE = STANFORD_DOGS_ROOT / "lists.tar"
IMAGES_DIR = STANFORD_DOGS_ROOT / "Images"
LISTS_DIR = STANFORD_DOGS_ROOT / "lists"


def download_if_needed(url: str, target_path: Path):
    if target_path.exists():
        print(f"Already downloaded: {target_path.name}")
        return
    print(f"Downloading {url} -> {target_path}")
    urllib.request.urlretrieve(url, target_path)


def extract_tar_if_needed(archive_path: Path, target_dir: Path, expected_path: Path):
    if expected_path.exists():
        print(f"Already extracted: {expected_path}")
        return
    print(f"Extracting {archive_path.name}...")
    with tarfile.open(archive_path) as tar:
        tar.extractall(target_dir)


def find_existing(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find any of: {candidates}")


def matlab_str(x) -> str:
    while isinstance(x, np.ndarray):
        x = x.squeeze()
        if x.ndim == 0:
            x = x.item()
            break
        x = x[0]
    return str(x)


def matlab_int(x) -> int:
    return int(np.array(x).squeeze())


download_if_needed(IMAGES_URL, IMAGES_ARCHIVE)
download_if_needed(LISTS_URL, LISTS_ARCHIVE)
extract_tar_if_needed(IMAGES_ARCHIVE, STANFORD_DOGS_ROOT, IMAGES_DIR)
extract_tar_if_needed(LISTS_ARCHIVE, STANFORD_DOGS_ROOT, LISTS_DIR)

TRAIN_LIST_MAT = find_existing(STANFORD_DOGS_ROOT / "train_list.mat", LISTS_DIR / "train_list.mat")
TEST_LIST_MAT = find_existing(STANFORD_DOGS_ROOT / "test_list.mat", LISTS_DIR / "test_list.mat")

train_list_data = loadmat(TRAIN_LIST_MAT)
test_list_data = loadmat(TEST_LIST_MAT)

train_rel_paths = [matlab_str(x) for x in train_list_data["file_list"].ravel()]
train_labels = [matlab_int(x) - 1 for x in train_list_data["labels"].ravel()]
test_rel_paths = [matlab_str(x) for x in test_list_data["file_list"].ravel()]
test_labels = [matlab_int(x) - 1 for x in test_list_data["labels"].ravel()]

label_to_folder = {}
for rel_path, label in list(zip(train_rel_paths, train_labels)) + list(zip(test_rel_paths, test_labels)):
    label_to_folder.setdefault(label, Path(rel_path).parts[0])

CLASS_NAMES = [label_to_folder[i].split("-", 1)[1].replace("_", " ") if "-" in label_to_folder[i] else label_to_folder[i].replace("_", " ") for i in range(len(label_to_folder))]
NUM_CLASSES = len(CLASS_NAMES)
print("Classes:", NUM_CLASSES)
print("Official train samples:", len(train_rel_paths))
print("Official test samples:", len(test_rel_paths))


In [ ]:
def build_metadata(rel_paths, labels, split_name: str) -> pd.DataFrame:
    rows = []
    for rel_path, label in zip(rel_paths, labels):
        image_path = IMAGES_DIR / rel_path
        with Image.open(image_path) as img:
            width, height = img.size
        folder_name = Path(rel_path).parts[0]
        breed_name = CLASS_NAMES[int(label)]
        rows.append(
            {
                "image_id": Path(rel_path).stem,
                "image_path": str(image_path),
                "relative_path": rel_path,
                "label": int(label),
                "class_name": breed_name,
                "folder_name": folder_name,
                "official_split": split_name,
                "width": width,
                "height": height,
                "aspect_ratio": width / height,
            }
        )
    return pd.DataFrame(rows)


train_meta_full = build_metadata(train_rel_paths, train_labels, "official_train")
test_meta = build_metadata(test_rel_paths, test_labels, "official_test")
meta = pd.concat([train_meta_full, test_meta], ignore_index=True)

summary_df = pd.DataFrame(
    [
        ("dataset", "Stanford Dogs"),
        ("total_images", len(meta)),
        ("official_train", len(train_meta_full)),
        ("official_test", len(test_meta)),
        ("num_classes", meta["label"].nunique()),
        ("min_images_per_class", int(meta.groupby("class_name").size().min())),
        ("max_images_per_class", int(meta.groupby("class_name").size().max())),
        ("mean_images_per_class", round(float(meta.groupby("class_name").size().mean()), 2)),
    ],
    columns=["metric", "value"],
)
display(summary_df)


### Richer EDA: image quality, breed distribution, and size variability

In addition to standard dataset counts, the following cells inspect image quality, breed distribution,
and resolution variability so the report can discuss why **Stanford Dogs** is a suitable fine-grained classification dataset.


In [ ]:
quality_rows = []
for image_path in tqdm(meta['image_path'], desc='Computing image-quality metadata'):
    with Image.open(image_path).convert('RGB') as img:
        gray = img.convert('L')
        hsv = img.convert('HSV')
        rgb_arr = np.asarray(img).astype(np.float32) / 255.0
        gray_arr = np.asarray(gray).astype(np.float32) / 255.0
        hsv_arr = np.asarray(hsv).astype(np.float32) / 255.0

        quality_rows.append({
            'brightness_mean': float(gray_arr.mean()),
            'contrast_std': float(gray_arr.std()),
            'saturation_mean': float(hsv_arr[..., 1].mean()),
            'r_mean': float(rgb_arr[..., 0].mean()),
            'g_mean': float(rgb_arr[..., 1].mean()),
            'b_mean': float(rgb_arr[..., 2].mean()),
        })

quality_df = pd.DataFrame(quality_rows)
meta = pd.concat([meta.reset_index(drop=True), quality_df], axis=1)
meta.to_csv(EDA_ARTIFACT_ROOT / 'metadata_with_quality.csv', index=False)

quality_summary = meta[[
    'brightness_mean', 'contrast_std', 'saturation_mean',
    'r_mean', 'g_mean', 'b_mean'
]].describe().T
display(quality_summary)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

sns.histplot(meta['brightness_mean'], bins=30, kde=True, ax=axes[0], color=BRIGHTNESS_COLOR, edgecolor='white', alpha=0.9)
axes[0].axvline(meta['brightness_mean'].mean(), color='black', linestyle='--', linewidth=1.6, label=f"mean={meta['brightness_mean'].mean():.2f}")
axes[0].set_title('Brightness distribution')
axes[0].set_xlabel('Mean grayscale brightness')
axes[0].legend()

sns.histplot(meta['contrast_std'], bins=30, kde=True, ax=axes[1], color=CONTRAST_COLOR, edgecolor='white', alpha=0.9)
axes[1].axvline(meta['contrast_std'].mean(), color='black', linestyle='--', linewidth=1.6, label=f"mean={meta['contrast_std'].mean():.2f}")
axes[1].set_title('Contrast distribution')
axes[1].set_xlabel('Std of grayscale intensity')
axes[1].legend()

sns.histplot(meta['saturation_mean'], bins=30, kde=True, ax=axes[2], color=SATURATION_COLOR, edgecolor='white', alpha=0.9)
axes[2].axvline(meta['saturation_mean'].mean(), color='black', linestyle='--', linewidth=1.6, label=f"mean={meta['saturation_mean'].mean():.2f}")
axes[2].set_title('Saturation distribution')
axes[2].set_xlabel('Mean saturation')
axes[2].legend()

for ax in axes:
    ax.grid(axis='y', alpha=0.25)

plt.tight_layout()
plt.savefig(EDA_ARTIFACT_ROOT / 'quality_distributions.png', bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
top_breeds = meta["class_name"].value_counts().head(6).index.tolist()
breed_subset = meta[meta["class_name"].isin(top_breeds)].copy()
sampled_meta = breed_subset.sample(min(len(breed_subset), 1200), random_state=SEED)
palette = dict(zip(top_breeds, sns.color_palette("Set2", n_colors=len(top_breeds))))

sns.boxplot(
    data=breed_subset,
    x="class_name",
    y="brightness_mean",
    hue="class_name",
    palette=palette,
    dodge=False,
    ax=axes[0],
    width=0.55,
    showfliers=False,
)
sns.stripplot(
    data=sampled_meta,
    x="class_name",
    y="brightness_mean",
    hue="class_name",
    palette=palette,
    dodge=False,
    ax=axes[0],
    alpha=0.15,
    size=2.2,
    jitter=0.22,
)
if axes[0].legend_ is not None:
    axes[0].legend_.remove()
axes[0].set_title("Brightness by selected breeds")
axes[0].set_xlabel("Breed")
axes[0].set_ylabel("Brightness")
axes[0].tick_params(axis="x", rotation=18)

sns.boxplot(
    data=breed_subset,
    x="class_name",
    y="saturation_mean",
    hue="class_name",
    palette=palette,
    dodge=False,
    ax=axes[1],
    width=0.55,
    showfliers=False,
)
sns.stripplot(
    data=sampled_meta,
    x="class_name",
    y="saturation_mean",
    hue="class_name",
    palette=palette,
    dodge=False,
    ax=axes[1],
    alpha=0.15,
    size=2.2,
    jitter=0.22,
)
if axes[1].legend_ is not None:
    axes[1].legend_.remove()
axes[1].set_title("Saturation by selected breeds")
axes[1].set_xlabel("Breed")
axes[1].set_ylabel("Saturation")
axes[1].tick_params(axis="x", rotation=18)

plt.tight_layout()
plt.savefig(EDA_ARTIFACT_ROOT / "breed_quality_boxplots.png", bbox_inches="tight")
plt.show()


In [ ]:
rgb_channel_summary = pd.DataFrame(
    {
        "channel": ["R", "G", "B"],
        "mean": [meta["r_mean"].mean(), meta["g_mean"].mean(), meta["b_mean"].mean()],
        "std_of_image_means": [meta["r_mean"].std(), meta["g_mean"].std(), meta["b_mean"].std()],
    }
)
display(rgb_channel_summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

bars_0 = axes[0].bar(
    rgb_channel_summary["channel"],
    rgb_channel_summary["mean"],
    color=RGB_COLORS,
    edgecolor="black",
    linewidth=0.8,
)
axes[0].set_title("Average RGB channel intensity")
axes[0].set_ylim(0, 1.0)
axes[0].set_xlabel("Channel")
axes[0].set_ylabel("Mean value")
axes[0].bar_label(bars_0, fmt="%.3f", padding=3, fontsize=10)
axes[0].grid(axis="y", alpha=0.25)

bars_1 = axes[1].bar(
    rgb_channel_summary["channel"],
    rgb_channel_summary["std_of_image_means"],
    color=RGB_COLORS,
    edgecolor="black",
    linewidth=0.8,
)
axes[1].set_title("Variation of per-image RGB means")
axes[1].set_xlabel("Channel")
axes[1].set_ylabel("Std of image means")
axes[1].bar_label(bars_1, fmt="%.3f", padding=3, fontsize=10)
axes[1].grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig(EDA_ARTIFACT_ROOT / "rgb_channel_summary.png", bbox_inches="tight")
plt.show()


In [ ]:
def show_extreme_examples(frame: pd.DataFrame, column: str, ascending: bool, title: str, save_name: str):
    sample = frame.sort_values(column, ascending=ascending).head(8).reset_index(drop=True)
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for ax, (_, row) in zip(axes.flat, sample.iterrows()):
        with Image.open(row["image_path"]) as img:
            ax.imshow(img)
        ax.set_title(f"{row['class_name']}\n{column}={row[column]:.2f}", fontsize=9)
        ax.axis("off")
    plt.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.savefig(EDA_ARTIFACT_ROOT / save_name, bbox_inches="tight")
    plt.show()


show_extreme_examples(meta, "brightness_mean", True, "Darkest images in the dataset", "darkest_examples.png")
show_extreme_examples(meta, "brightness_mean", False, "Brightest images in the dataset", "brightest_examples.png")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5.8))

split_counts = meta["official_split"].value_counts().reindex(["official_train", "official_test"])
axes[0].bar(
    ["Train", "Test"],
    split_counts.values,
    color=sns.color_palette("Set2", n_colors=2),
    edgecolor="black",
    linewidth=0.8,
)
axes[0].set_title("Official split sizes")
axes[0].set_xlabel("Split")
axes[0].set_ylabel("Images")
axes[0].grid(axis="y", alpha=0.25)

class_counts = meta.groupby("class_name").size().sort_values(ascending=False).head(20)
axes[1].bar(
    class_counts.index,
    class_counts.values,
    color=sns.color_palette("deep", n_colors=len(class_counts)),
    edgecolor="black",
    linewidth=0.3,
)
axes[1].set_title("Top 20 breeds by image count")
axes[1].set_xlabel("Breed")
axes[1].set_ylabel("Images")
axes[1].tick_params(axis="x", rotation=90, labelsize=8)
axes[1].grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig(EDA_ARTIFACT_ROOT / "dataset_distributions.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

sns.histplot(meta['width'], bins=30, kde=True, ax=axes[0], color=WIDTH_COLOR, edgecolor='white', alpha=0.9)
axes[0].axvline(meta['width'].mean(), color='black', linestyle='--', linewidth=1.6, label=f"mean={meta['width'].mean():.1f}")
axes[0].set_title('Image width distribution')
axes[0].set_xlabel('Width (pixels)')
axes[0].legend()

sns.histplot(meta['height'], bins=30, kde=True, ax=axes[1], color=HEIGHT_COLOR, edgecolor='white', alpha=0.9)
axes[1].axvline(meta['height'].mean(), color='black', linestyle='--', linewidth=1.6, label=f"mean={meta['height'].mean():.1f}")
axes[1].set_title('Image height distribution')
axes[1].set_xlabel('Height (pixels)')
axes[1].legend()

sns.histplot(meta['aspect_ratio'], bins=30, kde=True, ax=axes[2], color=ASPECT_COLOR, edgecolor='white', alpha=0.9)
axes[2].axvline(meta['aspect_ratio'].mean(), color='black', linestyle='--', linewidth=1.6, label=f"mean={meta['aspect_ratio'].mean():.2f}")
axes[2].set_title('Aspect ratio distribution')
axes[2].set_xlabel('Width / height')
axes[2].legend()

plt.tight_layout()
plt.savefig(EDA_ARTIFACT_ROOT / 'image_size_distribution.png', bbox_inches='tight')
plt.show()


In [ ]:
sampled = (
    meta.groupby("class_name", group_keys=False)
    .apply(lambda g: g.sample(1, random_state=SEED))
    .sample(12, random_state=SEED)
    .reset_index(drop=True)
)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, (_, row) in zip(axes.flat, sampled.iterrows()):
    with Image.open(row["image_path"]) as img:
        ax.imshow(img)
    ax.set_title(
        f"{row['class_name']}\n{row['official_split'].replace('official_', '')}",
        fontsize=9,
    )
    ax.axis("off")
plt.tight_layout()
plt.savefig(EDA_ARTIFACT_ROOT / "random_class_samples.png", bbox_inches="tight")
plt.show()


In [ ]:
splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=VAL_FROM_TRAIN_RATIO,
    random_state=SEED,
)
train_idx, val_idx = next(splitter.split(train_meta_full, train_meta_full["label"]))

train_meta = train_meta_full.iloc[train_idx].reset_index(drop=True)
val_meta = train_meta_full.iloc[val_idx].reset_index(drop=True)
test_meta = test_meta.reset_index(drop=True)

train_meta["split"] = "train"
val_meta["split"] = "val"
test_meta["split"] = "test"
split_meta = pd.concat([train_meta, val_meta, test_meta], ignore_index=True)
split_meta.to_csv(EDA_ARTIFACT_ROOT / "split_metadata.csv", index=False)

split_summary = split_meta.groupby("split").agg(
    images=("image_path", "count"),
    classes=("label", "nunique"),
    min_class_count=("label", lambda s: s.value_counts().min()),
    max_class_count=("label", lambda s: s.value_counts().max()),
)
display(split_summary)
assert len(train_meta) >= 5000


**EDA report note:** this dataset satisfies the assignment constraints well because the official training split already
contains `12,000` images, the benchmark has `120` dog breeds, and the task is a fine-grained classification problem with
significant appearance overlap between similar breeds.


## 2. Dataset, DataLoader, and Augmentation Setup

We keep the split fixed for both models and only change the training augmentation policy.
Both models use ImageNet normalization because they start from ImageNet-pretrained weights.


In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

resnet_train_transform = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.72, 1.0)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        transforms.RandomErasing(p=0.10),
    ]
)

vit_train_transform = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.78, 1.0)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.10),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        transforms.RandomErasing(p=0.08),
    ]
)

common_eval_transform = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)


class StanfordDogsFrameDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform=None):
        self.frame = frame.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        label = int(row["label"])
        if self.transform is not None:
            image = self.transform(image)
        return image, label


def build_loaders(train_transform):
    train_ds = StanfordDogsFrameDataset(train_meta, transform=train_transform)
    val_ds = StanfordDogsFrameDataset(val_meta, transform=common_eval_transform)
    test_ds = StanfordDogsFrameDataset(test_meta, transform=common_eval_transform)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )
    return train_loader, val_loader, test_loader


resnet_train_loader, resnet_val_loader, resnet_test_loader = build_loaders(resnet_train_transform)
vit_train_loader, vit_val_loader, vit_test_loader = build_loaders(vit_train_transform)
print("ResNet loaders:", len(resnet_train_loader), len(resnet_val_loader), len(resnet_test_loader))
print("ViT loaders:", len(vit_train_loader), len(vit_val_loader), len(vit_test_loader))


In [ ]:
def denormalize_image(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return torch.clamp(tensor.cpu() * std + mean, 0, 1)


images, labels = next(iter(resnet_train_loader))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, image, label in zip(axes.flat, images[:8], labels[:8]):
    ax.imshow(denormalize_image(image).permute(1, 2, 0))
    ax.set_title(CLASS_NAMES[int(label)], fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.savefig(EDA_ARTIFACT_ROOT / "augmented_batch_preview.png", bbox_inches="tight")
plt.show()


### Preprocessing rationale for the report

The preprocessing design is chosen to match both the dog-breed domain and the pretrained backbones:
- `Resize + crop` adapts variable-size dog photos to the fixed `224 x 224` input expected by ImageNet-pretrained models.
- `ImageNet normalization` keeps the input distribution compatible with pretrained weights.
- `ResNet` receives slightly stronger geometric and appearance augmentation.
- `ViT` keeps augmentation moderate so that fine-grained breed cues remain stable during training.


In [ ]:
preprocessing_summary = pd.DataFrame([
    {'component': 'Resize / crop', 'design': 'Resize to 256 then crop to 224', 'reason': 'Match pretrained backbone input size while preserving the main dog region clearly enough for classification'},
    {'component': 'Normalization', 'design': 'ImageNet mean/std', 'reason': 'Compatible with pretrained ResNet-18 and ViT-B/16 weights'},
    {'component': 'ResNet augmentation', 'design': 'RandomResizedCrop + flip + rotation + color jitter + erasing', 'reason': 'Improve robustness to pose, scale, illumination, and partial occlusion in dog photos'},
    {'component': 'ViT augmentation', 'design': 'Milder crop + flip + color jitter + erasing', 'reason': 'Keep augmentation moderate while still improving generalization on fine-grained breed differences'},
    {'component': 'Split strategy', 'design': 'Official train/test split with internal stratified validation split from train only', 'reason': 'Preserve the benchmark test set while keeping the training pool comfortably above the assignment size threshold'},
])
display(preprocessing_summary)


## 3. Model Building, Training, Evaluation, and Comparison

The notebook uses custom PyTorch loops and tracks:
- loss
- accuracy
- macro F1
- weighted F1
- confusion matrix
- classification report
- calibration (ECE + reliability diagram)


In [ ]:
def overlay_heatmap_on_image(image_np: np.ndarray, heatmap: np.ndarray, alpha: float = 0.38, cmap_name: str = "inferno"):
    cmap = mpl_cm.get_cmap(cmap_name)
    color_map = cmap(np.clip(heatmap, 0, 1))[..., :3]
    overlay = np.clip((1 - alpha) * image_np + alpha * color_map, 0, 1)
    return overlay


def plot_misclassified_gallery(frame, targets, preds, probs, title, save_path: Path, top_k: int = 12):
    mistakes = np.where(targets != preds)[0]
    if len(mistakes) == 0:
        print(f"No misclassified samples found for {title}.")
        return

    confidence = probs[mistakes].max(axis=1)
    chosen = mistakes[np.argsort(-confidence)[: min(top_k, len(mistakes))]]
    cols = 4
    rows = math.ceil(len(chosen) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4.5 * rows))
    axes = np.array(axes).reshape(-1)
    eval_frame = frame.reset_index(drop=True)

    for ax, idx in zip(axes, chosen):
        row = eval_frame.iloc[idx]
        with Image.open(row["image_path"]).convert("RGB") as img:
            ax.imshow(img)
        ax.set_title(
            f"True: {CLASS_NAMES[int(targets[idx])]}\nPred: {CLASS_NAMES[int(preds[idx])]}\nConf: {probs[idx].max():.2f}",
            fontsize=10,
        )
        ax.axis("off")

    for ax in axes[len(chosen):]:
        ax.axis("off")

    plt.suptitle(title, y=1.02, fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=220, bbox_inches="tight")
    plt.show()


def compute_gradcam_resnet(model, image_tensor, class_idx=None, target_layer=None):
    model.eval()
    if target_layer is None:
        target_layer = model.layer4[-1].conv2

    activations = []
    gradients = []

    def forward_hook(_, __, output):
        activations.append(output.detach())

    def backward_hook(_, grad_input, grad_output):
        gradients.append(grad_output[0].detach())

    handle_fwd = target_layer.register_forward_hook(forward_hook)
    handle_bwd = target_layer.register_full_backward_hook(backward_hook)

    input_batch = image_tensor.unsqueeze(0).to(DEVICE)
    output = model(input_batch)
    if class_idx is None:
        class_idx = int(output.argmax(dim=1).item())

    score = output[0, class_idx]
    model.zero_grad(set_to_none=True)
    score.backward()

    handle_fwd.remove()
    handle_bwd.remove()

    grad = gradients[0]
    act = activations[0]
    weights = grad.mean(dim=(2, 3), keepdim=True)
    cam = (weights * act).sum(dim=1, keepdim=True)
    cam = F.relu(cam)
    cam = F.interpolate(cam, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False)
    cam = cam[0, 0]
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-8)
    return cam.cpu().numpy(), class_idx


def extract_vit_last_attention_map(model, image_tensor):
    model.eval()
    x = image_tensor.unsqueeze(0).to(DEVICE)
    x = model._process_input(x)
    n = x.shape[0]
    batch_class_token = model.class_token.expand(n, -1, -1)
    x = torch.cat([batch_class_token, x], dim=1)
    x = x + model.encoder.pos_embedding
    x = model.encoder.dropout(x)

    attn_weights = None
    last_idx = len(model.encoder.layers) - 1

    for idx, block in enumerate(model.encoder.layers):
        y = block.ln_1(x)
        if idx == last_idx:
            y_attn, attn_weights = block.self_attention(y, y, y, need_weights=True, average_attn_weights=False)
        else:
            y_attn, _ = block.self_attention(y, y, y, need_weights=False)
        x = x + block.dropout(y_attn)
        y2 = block.ln_2(x)
        y2 = block.mlp(y2)
        x = x + y2

    cls_attn = attn_weights[0].mean(dim=0)[0, 1:]
    grid_size = int(math.sqrt(len(cls_attn)))
    attn_map = cls_attn.reshape(grid_size, grid_size)
    attn_map = attn_map / (attn_map.max() + 1e-8)
    attn_map = F.interpolate(attn_map.unsqueeze(0).unsqueeze(0), size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False)
    return attn_map[0, 0].detach().cpu().numpy()


def plot_interpretability_gallery(frame, model, transform, method: str, title: str, save_path: Path, sample_count: int = 6):
    sample_df = frame.sample(min(sample_count, len(frame)), random_state=SEED).reset_index(drop=True)
    cols = 3
    rows = math.ceil(len(sample_df) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(15, 4.8 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, sample_df.iterrows()):
        pil_img = Image.open(row["image_path"]).convert("RGB")
        input_tensor = transform(pil_img)
        image_np = np.asarray(pil_img.resize((IMAGE_SIZE, IMAGE_SIZE))).astype(np.float32) / 255.0

        with torch.no_grad():
            logits = model(input_tensor.unsqueeze(0).to(DEVICE))
            pred_idx = int(logits.argmax(dim=1).item())

        if method == "gradcam":
            heatmap, pred_idx = compute_gradcam_resnet(model, input_tensor, class_idx=pred_idx)
            overlay = overlay_heatmap_on_image(image_np, heatmap, alpha=0.42, cmap_name="inferno")
        else:
            heatmap = extract_vit_last_attention_map(model, input_tensor)
            overlay = overlay_heatmap_on_image(image_np, heatmap, alpha=0.40, cmap_name="viridis")

        ax.imshow(overlay)
        ax.set_title(
            f"True: {row['class_name']}\nPred: {CLASS_NAMES[pred_idx]}",
            fontsize=10,
        )
        ax.axis("off")

    for ax in axes[len(sample_df):]:
        ax.axis("off")

    plt.suptitle(title, y=1.02, fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=220, bbox_inches="tight")
    plt.show()


def freeze_resnet_backbone(model):
    for param in model.parameters():
        param.requires_grad = False
    for param in model.fc.parameters():
        param.requires_grad = True


def freeze_vit_backbone(model):
    for param in model.parameters():
        param.requires_grad = False
    for param in model.heads.parameters():
        param.requires_grad = True


def run_simple_experiment(model_builder, model_label, train_loader, val_loader, test_loader, epochs, lr, weight_decay, freeze_fn=None):
    model = model_builder(NUM_CLASSES).to(DEVICE)
    if freeze_fn is not None:
        freeze_fn(model)
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=weight_decay)
    else:
        trainable_params = count_total_params(model)
        optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    scheduler = CosineAnnealingLR(optimizer, T_max=max(1, epochs))
    criterion = nn.CrossEntropyLoss()
    _, training_seconds = fit_model(model, train_loader, val_loader, criterion, optimizer, scheduler=scheduler, epochs=epochs, stage_name=model_label)
    targets, preds, probs = predict_model(model, test_loader)
    ece, *_ = compute_ece(probs, targets, n_bins=10)
    return {
        "setting": model_label,
        "accuracy": accuracy_score(targets, preds),
        "macro_f1": f1_score(targets, preds, average="macro"),
        "weighted_f1": f1_score(targets, preds, average="weighted"),
        "ece": ece,
        "train_time_s": training_seconds,
        "trainable_params": trainable_params,
    }


In [ ]:
def count_total_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def create_resnet18(num_classes: int) -> nn.Module:
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def create_vit_b16(num_classes: int) -> nn.Module:
    model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
    model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    return model


def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    all_targets, all_preds = [], []

    for images, targets in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        if is_train:
            optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, targets)
        if is_train:
            loss.backward()
            optimizer.step()
        preds = outputs.argmax(dim=1)
        total_loss += loss.item() * images.size(0)
        all_targets.extend(targets.detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())

    return {
        'loss': total_loss / len(loader.dataset),
        'acc': accuracy_score(all_targets, all_preds),
        'macro_f1': f1_score(all_targets, all_preds, average='macro'),
        'weighted_f1': f1_score(all_targets, all_preds, average='weighted'),
    }


def fit_model(model, train_loader, val_loader, criterion, optimizer, scheduler=None, epochs=10, stage_name='train'):
    history = []
    best_val_acc = -1.0
    best_state = copy.deepcopy(model.state_dict())
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        train_metrics = run_epoch(model, train_loader, criterion, optimizer=optimizer)
        val_metrics = run_epoch(model, val_loader, criterion, optimizer=None)
        if scheduler is not None:
            scheduler.step()

        history.append({
            'stage': stage_name,
            'epoch': epoch,
            'train_loss': train_metrics['loss'],
            'train_acc': train_metrics['acc'],
            'val_loss': val_metrics['loss'],
            'val_acc': val_metrics['acc'],
            'val_macro_f1': val_metrics['macro_f1'],
            'lr': optimizer.param_groups[0]['lr'],
        })
        print(f"[{stage_name}] epoch {epoch:02d}/{epochs} | train_acc={train_metrics['acc']:.4f} | val_acc={val_metrics['acc']:.4f}")
        if val_metrics['acc'] > best_val_acc:
            best_val_acc = val_metrics['acc']
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return pd.DataFrame(history), time.time() - start_time


def predict_model(model, loader):
    model.eval()
    all_targets, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for images, targets in tqdm(loader, leave=False):
            images = images.to(DEVICE, non_blocking=True)
            outputs = model(images)
            probs = outputs.softmax(dim=1)
            preds = probs.argmax(dim=1)
            all_targets.extend(targets.numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.append(probs.cpu())
    return np.array(all_targets), np.array(all_preds), torch.cat(all_probs, dim=0).numpy()


def evaluate_model(model, loader, class_names, model_name, artifact_dir):
    targets, preds, probs = predict_model(model, loader)
    report = classification_report(targets, preds, target_names=class_names, zero_division=0, output_dict=True)
    report_df = pd.DataFrame(report).transpose()
    cm = confusion_matrix(targets, preds)
    cm_norm = confusion_matrix(targets, preds, normalize='true')
    safe_name = model_name.lower().replace('/', '_').replace('-', '_').replace(' ', '_')

    fig, ax = plt.subplots(figsize=(23, 19))
    sns.heatmap(
        cm,
        cmap=CONFUSION_CMAP,
        annot=True,
        fmt='d',
        annot_kws={'size': 5.2},
        linewidths=0.15,
        linecolor='white',
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={'shrink': 0.8},
        ax=ax,
    )
    ax.set_title(f'{model_name} confusion matrix (counts)', pad=16)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.tick_params(axis='x', labelrotation=90, labelsize=7)
    ax.tick_params(axis='y', labelrotation=0, labelsize=7)
    plt.tight_layout()
    fig.savefig(artifact_dir / f'{safe_name}_confusion_matrix_counts.png', dpi=260, bbox_inches='tight')
    plt.show()

    fig, ax = plt.subplots(figsize=(23, 19))
    sns.heatmap(
        cm_norm,
        cmap=HEATMAP_CMAP,
        annot=True,
        fmt='.2f',
        annot_kws={'size': 5.2},
        linewidths=0.15,
        linecolor='white',
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={'shrink': 0.8},
        vmin=0,
        vmax=max(0.3, float(cm_norm.max())),
        ax=ax,
    )
    ax.set_title(f'{model_name} confusion matrix (row-normalized)', pad=16)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.tick_params(axis='x', labelrotation=90, labelsize=7)
    ax.tick_params(axis='y', labelrotation=0, labelsize=7)
    plt.tight_layout()
    fig.savefig(artifact_dir / f'{safe_name}_confusion_matrix_normalized.png', dpi=260, bbox_inches='tight')
    plt.show()

    off_diag = []
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if i != j and cm[i, j] > 0:
                off_diag.append({
                    'true_class': class_names[i],
                    'pred_class': class_names[j],
                    'count': int(cm[i, j]),
                    'row_normalized': float(cm_norm[i, j]),
                })
    top_confusions = pd.DataFrame(off_diag).sort_values(['count', 'row_normalized'], ascending=False).head(20)
    top_confusions.to_csv(artifact_dir / f'{safe_name}_top_confusions.csv', index=False)

    metrics = {
        'model': model_name,
        'accuracy': accuracy_score(targets, preds),
        'macro_f1': f1_score(targets, preds, average='macro'),
        'weighted_f1': f1_score(targets, preds, average='weighted'),
    }
    report_df.to_csv(artifact_dir / f'{safe_name}_classification_report.csv')
    with open(artifact_dir / f'{safe_name}_metrics.json', 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2)
    return metrics, report_df, targets, preds, probs


def compute_ece(probs: np.ndarray, targets: np.ndarray, n_bins: int = 10):
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    correctness = (predictions == targets).astype(np.float32)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    bin_acc, bin_conf, bin_count = [], [], []
    for left, right in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (confidences > left) & (confidences <= right)
        if left == 0:
            mask = (confidences >= left) & (confidences <= right)
        count = int(mask.sum())
        bin_count.append(count)
        if count == 0:
            bin_acc.append(0.0)
            bin_conf.append(0.0)
            continue
        acc = float(correctness[mask].mean())
        conf = float(confidences[mask].mean())
        bin_acc.append(acc)
        bin_conf.append(conf)
        ece += abs(acc - conf) * (count / len(targets))
    return float(ece), np.array(bin_acc), np.array(bin_conf), np.array(bin_count), bin_edges


def plot_reliability_diagram(ece, bin_acc, bin_conf, bin_count, bin_edges, title, save_path: Path):
    centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    widths = np.diff(bin_edges)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
    axes[0].bar(centers, bin_acc, width=widths, alpha=0.78, edgecolor='white', color=MODEL_PALETTE[1], label='Accuracy')
    axes[0].plot([0, 1], [0, 1], '--', color='black', linewidth=1.5, label='Perfect calibration')
    axes[0].plot(centers, bin_conf, marker='o', markersize=6, color=MODEL_PALETTE[0], linewidth=2, label='Confidence')
    axes[0].set_title(f'{title} reliability diagram\nECE = {ece:.4f}')
    axes[0].set_xlabel('Confidence bin')
    axes[0].set_ylabel('Accuracy / confidence')
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.25)

    axes[1].bar(centers, bin_count, width=widths, color=QUALITY_PALETTE[2], edgecolor='white')
    axes[1].set_title(f'{title} confidence histogram')
    axes[1].set_xlabel('Confidence bin')
    axes[1].set_ylabel('Sample count')
    axes[1].grid(axis='y', alpha=0.25)

    plt.tight_layout()
    fig.savefig(save_path, dpi=220, bbox_inches='tight')
    plt.show()


def save_checkpoint(path: Path, model: nn.Module, history_df: pd.DataFrame, config: dict):
    torch.save(
        {
            'model_state_dict': model.state_dict(),
            'history': history_df.to_dict(orient='records'),
            'config': config,
            'class_names': CLASS_NAMES,
        },
        path,
    )
    print('Saved checkpoint to', path)


### 3.1 CNN Model: ResNet-18


In [ ]:
RESNET_CKPT = MODEL_ROOT / "stanforddogs_resnet18.pth"
resnet_model = create_resnet18(NUM_CLASSES).to(DEVICE)
print("ResNet-18 params:", f"{count_total_params(resnet_model):,}")

if LOAD_EXISTING_CHECKPOINTS and RESNET_CKPT.exists():
    checkpoint = torch.load(RESNET_CKPT, map_location=DEVICE)
    resnet_model.load_state_dict(checkpoint["model_state_dict"])
    resnet_history = pd.DataFrame(checkpoint.get("history", []))
    resnet_training_seconds = checkpoint.get("config", {}).get("training_seconds", None)
    print("Loaded existing ResNet-18 checkpoint.")
else:
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(resnet_model.parameters(), lr=RESNET_LR, weight_decay=RESNET_WEIGHT_DECAY)
    warmup = LinearLR(optimizer, start_factor=0.2, total_iters=RESNET_WARMUP_EPOCHS)
    cosine = CosineAnnealingLR(optimizer, T_max=max(1, RESNET_EPOCHS - RESNET_WARMUP_EPOCHS))
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[RESNET_WARMUP_EPOCHS])
    resnet_history, resnet_training_seconds = fit_model(
        resnet_model, resnet_train_loader, resnet_val_loader, criterion, optimizer, scheduler, RESNET_EPOCHS, "stanforddogs_resnet18"
    )
    save_checkpoint(
        RESNET_CKPT,
        resnet_model,
        resnet_history,
        {"epochs": RESNET_EPOCHS, "learning_rate": RESNET_LR, "training_seconds": resnet_training_seconds},
    )

display(resnet_history.tail())


In [ ]:
resnet_metrics, resnet_report_df, resnet_targets, resnet_preds, resnet_probs = evaluate_model(
    resnet_model, resnet_test_loader, CLASS_NAMES, "ResNet-18", CNN_ARTIFACT_ROOT
)
display(pd.DataFrame([resnet_metrics]))
display(resnet_report_df[["precision", "recall", "f1-score", "support"]].head())

resnet_ece, resnet_bin_acc, resnet_bin_conf, resnet_bin_count, resnet_bin_edges = compute_ece(
    resnet_probs, resnet_targets, n_bins=10
)
plot_reliability_diagram(
    resnet_ece, resnet_bin_acc, resnet_bin_conf, resnet_bin_count, resnet_bin_edges,
    "ResNet-18", CNN_ARTIFACT_ROOT / "resnet18_calibration.png"
)
print("ResNet-18 ECE:", round(resnet_ece, 6))


### 3.2 Vision Transformer Model: ViT-B/16


In [ ]:
VIT_CKPT = MODEL_ROOT / "stanforddogs_vit_b16.pth"
vit_model = create_vit_b16(NUM_CLASSES).to(DEVICE)
print("ViT-B/16 params:", f"{count_total_params(vit_model):,}")

if LOAD_EXISTING_CHECKPOINTS and VIT_CKPT.exists():
    checkpoint = torch.load(VIT_CKPT, map_location=DEVICE)
    vit_model.load_state_dict(checkpoint["model_state_dict"])
    vit_history = pd.DataFrame(checkpoint.get("history", []))
    vit_training_seconds = checkpoint.get("config", {}).get("training_seconds", None)
    print("Loaded existing ViT-B/16 checkpoint.")
else:
    criterion = nn.CrossEntropyLoss()

    for param in vit_model.parameters():
        param.requires_grad = False
    for param in vit_model.heads.parameters():
        param.requires_grad = True

    optimizer_head = AdamW(filter(lambda p: p.requires_grad, vit_model.parameters()), lr=VIT_HEAD_LR, weight_decay=VIT_WEIGHT_DECAY)
    scheduler_head = CosineAnnealingLR(optimizer_head, T_max=max(1, VIT_HEAD_EPOCHS))
    vit_head_history, vit_head_seconds = fit_model(
        vit_model, vit_train_loader, vit_val_loader, criterion, optimizer_head, scheduler_head, VIT_HEAD_EPOCHS, "stanforddogs_vit_head"
    )

    for param in vit_model.parameters():
        param.requires_grad = True

    optimizer_full = AdamW(vit_model.parameters(), lr=VIT_FINETUNE_LR, weight_decay=VIT_WEIGHT_DECAY)
    scheduler_full = CosineAnnealingLR(optimizer_full, T_max=max(1, VIT_FINETUNE_EPOCHS))
    vit_full_history, vit_full_seconds = fit_model(
        vit_model, vit_train_loader, vit_val_loader, criterion, optimizer_full, scheduler_full, VIT_FINETUNE_EPOCHS, "stanforddogs_vit_full"
    )

    vit_training_seconds = vit_head_seconds + vit_full_seconds
    vit_history = pd.concat([vit_head_history, vit_full_history], ignore_index=True)
    vit_history["epoch_total"] = np.arange(1, len(vit_history) + 1)
    save_checkpoint(
        VIT_CKPT,
        vit_model,
        vit_history,
        {"head_epochs": VIT_HEAD_EPOCHS, "finetune_epochs": VIT_FINETUNE_EPOCHS, "training_seconds": vit_training_seconds},
    )

display(vit_history.tail())


In [ ]:
vit_metrics, vit_report_df, vit_targets, vit_preds, vit_probs = evaluate_model(
    vit_model, vit_test_loader, CLASS_NAMES, "ViT-B/16", VIT_ARTIFACT_ROOT
)
display(pd.DataFrame([vit_metrics]))
display(vit_report_df[["precision", "recall", "f1-score", "support"]].head())

vit_ece, vit_bin_acc, vit_bin_conf, vit_bin_count, vit_bin_edges = compute_ece(vit_probs, vit_targets, n_bins=10)
plot_reliability_diagram(
    vit_ece, vit_bin_acc, vit_bin_conf, vit_bin_count, vit_bin_edges,
    "ViT-B/16", VIT_ARTIFACT_ROOT / "vit_b16_calibration.png"
)
print("ViT-B/16 ECE:", round(vit_ece, 6))


## 4. Experimental Results, Figures, Analysis, and Discussion


In [ ]:
comparison_df = pd.DataFrame(
    [
        {
            "Model": "ResNet-18",
            "Family": "CNN",
            "Params": count_total_params(resnet_model),
            "Training strategy": f"Full fine-tuning for {RESNET_EPOCHS} epochs",
            "Test accuracy": resnet_metrics["accuracy"],
            "Macro F1": resnet_metrics["macro_f1"],
            "Weighted F1": resnet_metrics["weighted_f1"],
            "ECE": resnet_ece,
            "Train time (s)": resnet_training_seconds,
        },
        {
            "Model": "ViT-B/16",
            "Family": "Transformer",
            "Params": count_total_params(vit_model),
            "Training strategy": f"Head {VIT_HEAD_EPOCHS} + full fine-tune {VIT_FINETUNE_EPOCHS} epochs",
            "Test accuracy": vit_metrics["accuracy"],
            "Macro F1": vit_metrics["macro_f1"],
            "Weighted F1": vit_metrics["weighted_f1"],
            "ECE": vit_ece,
            "Train time (s)": vit_training_seconds,
        },
    ]
)

comparison_df["Params(M)"] = comparison_df["Params"] / 1_000_000
comparison_df.to_csv(ARTIFACT_ROOT / "model_comparison.csv", index=False)
display(comparison_df)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))
models_order = comparison_df["Model"].tolist()

bars_acc = axes[0].bar(models_order, comparison_df["Test accuracy"], color=MODEL_PALETTE, edgecolor="black", linewidth=0.8)
axes[0].set_title("Test accuracy")
axes[0].set_ylim(0, 1.0)
axes[0].bar_label(bars_acc, fmt="%.3f", padding=3, fontsize=10)
axes[0].grid(axis="y", alpha=0.25)

bars_f1 = axes[1].bar(models_order, comparison_df["Macro F1"], color=MODEL_PALETTE, edgecolor="black", linewidth=0.8)
axes[1].set_title("Macro F1")
axes[1].set_ylim(0, 1.0)
axes[1].bar_label(bars_f1, fmt="%.3f", padding=3, fontsize=10)
axes[1].grid(axis="y", alpha=0.25)

bars_ece = axes[2].bar(models_order, comparison_df["ECE"], color=MODEL_PALETTE, edgecolor="black", linewidth=0.8)
axes[2].set_title("Calibration error")
axes[2].bar_label(bars_ece, fmt="%.4f", padding=3, fontsize=10)
axes[2].grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.savefig(ARTIFACT_ROOT / "comparison_overview.png", bbox_inches="tight")
plt.show()


### 5.1 Misclassified Examples Gallery

This gallery highlights the most confident mistakes made by each model. These examples are useful for the error-analysis
subsection of the report because they show which dog breeds remain visually confusing even after transfer learning.


In [ ]:
if RUN_MISCLASSIFIED_GALLERY:
    plot_misclassified_gallery(
        test_meta,
        resnet_targets,
        resnet_preds,
        resnet_probs,
        "ResNet-18 misclassified examples",
        CNN_ARTIFACT_ROOT / "resnet18_misclassified_gallery.png",
        top_k=MISCLASSIFIED_GALLERY_SAMPLES,
    )
    plot_misclassified_gallery(
        test_meta,
        vit_targets,
        vit_preds,
        vit_probs,
        "ViT-B/16 misclassified examples",
        VIT_ARTIFACT_ROOT / "vit_b16_misclassified_gallery.png",
        top_k=MISCLASSIFIED_GALLERY_SAMPLES,
    )
else:
    print("Set RUN_MISCLASSIFIED_GALLERY = True to export the misclassified examples galleries.")


### 5.2 Interpretability: Grad-CAM and ViT Attention

The goal here is to visualize *where* each model looks when producing a prediction:
- `Grad-CAM` for `ResNet-18`
- last-layer class-token attention visualization for `ViT-B/16`


In [ ]:
if RUN_INTERPRETABILITY:
    plot_interpretability_gallery(
        test_meta,
        resnet_model,
        common_eval_transform,
        method="gradcam",
        title="ResNet-18 Grad-CAM examples",
        save_path=CNN_ARTIFACT_ROOT / "resnet18_gradcam_gallery.png",
        sample_count=INTERPRETABILITY_SAMPLES,
    )
    plot_interpretability_gallery(
        test_meta,
        vit_model,
        common_eval_transform,
        method="attention",
        title="ViT-B/16 attention visualization examples",
        save_path=VIT_ARTIFACT_ROOT / "vit_b16_attention_gallery.png",
        sample_count=INTERPRETABILITY_SAMPLES,
    )
else:
    print("Set RUN_INTERPRETABILITY = True to generate Grad-CAM and attention-visualization outputs.")


### 5.3 Augmentation vs No Augmentation

This ablation compares the current augmentation pipeline against a no-augmentation baseline.
It is disabled by default because it requires additional training runs.


In [ ]:
no_aug_train_transform = common_eval_transform

if RUN_ABLATIONS:
    resnet_aug_loaders = build_loaders(resnet_train_transform)
    resnet_noaug_loaders = build_loaders(no_aug_train_transform)
    vit_aug_loaders = build_loaders(vit_train_transform)
    vit_noaug_loaders = build_loaders(no_aug_train_transform)

    augmentation_results = [
        run_simple_experiment(
            create_resnet18,
            "ResNet-18 | with augmentation",
            *resnet_aug_loaders,
            epochs=RESNET_ABLATION_EPOCHS,
            lr=RESNET_FULL_LR,
            weight_decay=RESNET_WEIGHT_DECAY,
        ),
        run_simple_experiment(
            create_resnet18,
            "ResNet-18 | no augmentation",
            *resnet_noaug_loaders,
            epochs=RESNET_ABLATION_EPOCHS,
            lr=RESNET_FULL_LR,
            weight_decay=RESNET_WEIGHT_DECAY,
        ),
        run_simple_experiment(
            create_vit_b16,
            "ViT-B/16 | with augmentation",
            *vit_aug_loaders,
            epochs=VIT_ABLATION_EPOCHS,
            lr=VIT_FULL_LR,
            weight_decay=VIT_WEIGHT_DECAY,
        ),
        run_simple_experiment(
            create_vit_b16,
            "ViT-B/16 | no augmentation",
            *vit_noaug_loaders,
            epochs=VIT_ABLATION_EPOCHS,
            lr=VIT_FULL_LR,
            weight_decay=VIT_WEIGHT_DECAY,
        ),
    ]

    augmentation_results_df = pd.DataFrame(augmentation_results)
    augmentation_results_df.to_csv(ARTIFACT_ROOT / "augmentation_ablation.csv", index=False)
    display(augmentation_results_df)
else:
    print("Set RUN_ABLATIONS = True to run the augmentation-vs-no-augmentation ablation.")


### 5.4 Freeze Backbone vs Full Fine-Tune

This ablation compares two optimization strategies for both models:
- freeze the backbone and train only the classifier head,
- full fine-tuning.


In [ ]:
if RUN_ABLATIONS:
    resnet_base_loaders = build_loaders(resnet_train_transform)
    vit_base_loaders = build_loaders(vit_train_transform)

    freeze_full_results = [
        run_simple_experiment(
            create_resnet18,
            "ResNet-18 | freeze backbone",
            *resnet_base_loaders,
            epochs=RESNET_ABLATION_EPOCHS,
            lr=RESNET_FREEZE_LR,
            weight_decay=RESNET_WEIGHT_DECAY,
            freeze_fn=freeze_resnet_backbone,
        ),
        run_simple_experiment(
            create_resnet18,
            "ResNet-18 | full fine-tune",
            *resnet_base_loaders,
            epochs=RESNET_ABLATION_EPOCHS,
            lr=RESNET_FULL_LR,
            weight_decay=RESNET_WEIGHT_DECAY,
        ),
        run_simple_experiment(
            create_vit_b16,
            "ViT-B/16 | freeze backbone",
            *vit_base_loaders,
            epochs=VIT_ABLATION_EPOCHS,
            lr=VIT_FREEZE_LR,
            weight_decay=VIT_WEIGHT_DECAY,
            freeze_fn=freeze_vit_backbone,
        ),
        run_simple_experiment(
            create_vit_b16,
            "ViT-B/16 | full fine-tune",
            *vit_base_loaders,
            epochs=VIT_ABLATION_EPOCHS,
            lr=VIT_FULL_LR,
            weight_decay=VIT_WEIGHT_DECAY,
        ),
    ]

    freeze_full_results_df = pd.DataFrame(freeze_full_results)
    freeze_full_results_df.to_csv(ARTIFACT_ROOT / "freeze_vs_full_ablation.csv", index=False)
    display(freeze_full_results_df)
else:
    print("Set RUN_ABLATIONS = True to run the freeze-vs-full-fine-tune ablation.")


## Final Report Checklist

After running this notebook, you should have enough material for:
- dataset choice and EDA
- preprocessing and augmentation
- ResNet-18 vs ViT-B/16 comparison
- tables, figures, and discussion
- calibration
- misclassified examples gallery
- Grad-CAM and ViT attention visualization
- augmentation-vs-no-augmentation ablation
- freeze-backbone vs full-fine-tune ablation

Outputs are saved under:
- `../artifacts/stanford_dogs/eda/`
- `../artifacts/stanford_dogs/cnn/`
- `../artifacts/stanford_dogs/vit/`
- `../artifacts/stanford_dogs/*.csv`
